In [0]:
# Databricks notebook source
# MAGIC %run ../00_setup/00_adls_gen2_oauth_connection

import os
import zipfile
import shutil

zip_abfss = "abfss://bronze@stspmobilitydev001.dfs.core.windows.net/raw/gtfs/cittamobi_gtfs.zip"
local_zip = "/tmp/cittamobi_gtfs.zip"
local_extract_dir = "/tmp/cittamobi_gtfs_extracted"
bronze_extract_path = "abfss://bronze@stspmobilitydev001.dfs.core.windows.net/gtfs/"

# limpeza local
if os.path.exists(local_extract_dir):
    shutil.rmtree(local_extract_dir)
os.makedirs(local_extract_dir, exist_ok=True)

# copiar do ADLS para local do driver
dbutils.fs.cp(zip_abfss, f"file:{local_zip}", True)

print("ZIP copiado para local:", local_zip)
print("Tamanho local (bytes):", os.path.getsize(local_zip))

# extrair
with zipfile.ZipFile(local_zip, "r") as zip_ref:
    zip_ref.extractall(local_extract_dir)

print("Arquivos extraídos localmente:")
for f in os.listdir(local_extract_dir):
    print("-", f)

# enviar arquivos extraídos para o bronze/gtfs
for file_name in os.listdir(local_extract_dir):
    local_file = os.path.join(local_extract_dir, file_name)
    dbutils.fs.cp(f"file:{local_file}", bronze_extract_path + file_name, True)

print("Arquivos enviados para:", bronze_extract_path)

display(dbutils.fs.ls(bronze_extract_path))